# Path E v3 — organism-level Mycoplasma whole-cell MIC query

v2 confirmed ChEMBL has zero protein-level Mycoplasma target records for the 11 UniProt orthologs of our 5 drug-target enzymes. v2 also enumerated the universe: ChEMBL has only 6 Mycoplasma targets total, all `target_type=ORGANISM` (whole-cell MIC assays where the assay was run on intact bacteria, not on purified protein).

v3 asks the natural follow-up: do those 6 organism-level targets have any activity records for our 4 ChEMBL-resolved compounds (Mupirocin, Tavaborole, REP3123, Auranofin)? If yes, those are whole-cell Mycoplasma MIC numbers — exactly what Path E wants, just at species-resolution rather than protein-resolution. If no, ChEMBL has zero Mycoplasma data on these compounds at any granularity, and the only path to MIC numbers is primary clinical-microbiology literature.

Wall: <1 min. 24 API calls (6 organism targets × 4 compounds).

In [ ]:
# Cell 1 — install + clone + PAT + repo refresh.
!pip install -q requests>=2.31
import os, subprocess, getpass
BRANCH = "claude/syn3a-whole-cell-simulator-REjHC"
REPO_URL = "https://github.com/Nikku03/cell.git"
REPO_DIR = "/content/cell"
def _run(cmd, cwd=None):
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if r.stdout.strip(): print(r.stdout.rstrip())
    if r.stderr.strip(): print(r.stderr.rstrip())
    if r.returncode != 0: raise RuntimeError(f"{cmd!r} exit {r.returncode}")
    return r
if not os.path.isdir(REPO_DIR):
    _run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR])
else:
    _run(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR)
    _run(["git", "checkout", BRANCH], cwd=REPO_DIR)
    _run(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd=REPO_DIR)
%cd /content/cell
if not os.environ.get("GITHUB_PAT", "").strip():
    pat = getpass.getpass("Paste your GitHub PAT (input hidden): ").strip()
    if not pat: raise ValueError("empty PAT")
    os.environ["GITHUB_PAT"] = pat

In [ ]:
# Cell 2 — re-discover the 6 organism-level Mycoplasma targets.
import requests, json, time
UA = {"User-Agent": "cell-sim-bot/1.0", "Accept": "application/json"}
CHEMBL = "https://www.ebi.ac.uk/chembl/api/data"

r = requests.get(f"{CHEMBL}/target.json",
                 params={"organism__icontains": "Mycoplasma", "limit": "50"},
                 headers=UA, timeout=30)
r.raise_for_status()
ORGANISM_TARGETS = r.json().get("targets", [])
print(f"{len(ORGANISM_TARGETS)} Mycoplasma organism-level targets in ChEMBL:")
for t in ORGANISM_TARGETS:
    print(f"  {t.get('target_chembl_id','?'):14s}  {t.get('organism','?')[:35]:35s}  {t.get('target_type','?')}")

In [ ]:
# Cell 3 — query activities for each (organism, compound) pair.
COMPOUNDS = [
    ("CHEMBL719",      "Mupirocin"),
    ("CHEMBL443052",   "Tavaborole"),
    ("CHEMBL4297370",  "REP3123"),
    ("CHEMBL1366",     "Auranofin"),
]

MATRIX = []
for tgt in ORGANISM_TARGETS:
    tid = tgt.get("target_chembl_id")
    org = tgt.get("organism")
    for cid, cname in COMPOUNDS:
        try:
            r = requests.get(
                f"{CHEMBL}/activity.json",
                params={"target_chembl_id": tid,
                        "molecule_chembl_id": cid,
                        "limit": "500"},
                headers=UA, timeout=30,
            )
            r.raise_for_status()
            data = r.json()
        except Exception as e:
            print(f"  {tid} x {cid}: FAIL {type(e).__name__}: {e}")
            continue
        acts = data.get("activities", [])
        if acts:
            print(f"\n*** HIT: {cname} x {org} ({tid}) -> {len(acts)} records")
            for a in acts[:5]:
                print(f"    {(a.get('standard_type') or '?'):10s}  "
                      f"{a.get('standard_value','?')} {a.get('standard_units','?')}  "
                      f"assay_org={a.get('assay_organism','?')[:30] if a.get('assay_organism') else '?'}  "
                      f"doc={a.get('document_chembl_id','?')}")
        else:
            print(f"  empty: {cname:12s} x {org[:30]:30s}  ({tid} x {cid})")
        MATRIX.append({
            "target_chembl_id": tid,
            "organism": org,
            "compound_chembl_id": cid,
            "compound_name": cname,
            "n_activities": len(acts),
            "activities": [{
                "standard_type": a.get("standard_type"),
                "standard_value": a.get("standard_value"),
                "standard_units": a.get("standard_units"),
                "assay_organism": a.get("assay_organism"),
                "assay_chembl_id": a.get("assay_chembl_id"),
                "document_chembl_id": a.get("document_chembl_id"),
                "pchembl_value": a.get("pchembl_value"),
            } for a in acts],
        })
        time.sleep(0.2)

n_hit = sum(1 for r in MATRIX if r["n_activities"] > 0)
print(f"\n=== summary: {n_hit} / {len(MATRIX)} (organism, compound) pairs have activity records ===")

In [ ]:
# Cell 4 — save + push.
from pathlib import Path
out = {
    "organism_targets": ORGANISM_TARGETS,
    "compounds": [{"chembl_id": cid, "name": cname} for cid, cname in COMPOUNDS],
    "matrix": MATRIX,
    "summary": {"n_pairs_total": len(MATRIX),
                "n_pairs_with_activity": sum(1 for r in MATRIX if r["n_activities"] > 0)},
}
out_path = Path("outputs/toxicity/path_e_mic_lookup_v3_organism_level.json")
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(json.dumps(out, indent=2, default=str))
print(f"wrote {out_path} ({out_path.stat().st_size} bytes)")

pat = os.environ.get("GITHUB_PAT", "").strip()
if not pat: raise SystemExit("GITHUB_PAT not set")
_run(["git", "config", "user.email", "cell-sim-bot@noreply.local"])
_run(["git", "config", "user.name", "cell-sim-bot"])
_run(["git", "add", "-f", str(out_path)])
status = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True)
if status.stdout.strip():
    _run(["git", "commit", "-m", "Session 27 v3: ChEMBL organism-level Mycoplasma whole-cell MIC matrix"])
    remote = f"https://{pat}@github.com/Nikku03/cell.git"
    _run(["git", "push", remote, BRANCH])
    print("\npush complete.")
else:
    print("nothing changed")